In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("spambase.data", header=None)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

X_train_gd = np.column_stack([np.ones(X_train_scaled.shape[0]), X_train_scaled])
X_test_gd  = np.column_stack([np.ones(X_test_scaled.shape[0]), X_test_scaled])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def cross_entropy_loss(X, y, theta):
    preds = sigmoid(X @ theta)
    eps = 1e-15
    preds = np.clip(preds, eps, 1 - eps)
    return -np.mean(y * np.log(preds) + (1 - y) * np.log(1 - preds))

def gradient_descent_logistic(X, y, alpha, num_iters, theta0=None):
    m, n = X.shape
    theta = np.zeros(n) if theta0 is None else theta0.copy()

    for _ in range(num_iters):
        preds = sigmoid(X @ theta)
        grad = (X.T @ (preds - y)) / m
        theta = theta - alpha * grad

    return theta

def classification_metrics(X, y, theta, threshold=0.5):
    probs = sigmoid(X @ theta)
    y_pred = (probs >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred, zero_division=0),
        "F1": f1_score(y, y_pred, zero_division=0)
    }

alphas = [0.01, 0.1, 0.5]
checkpoints = [10, 50, 100]

rows = []

for alpha in alphas:
    theta = np.zeros(X_train_gd.shape[1])
    prev_iter = 0

    for it in checkpoints:
        theta = gradient_descent_logistic(
            X_train_gd, Y_train, alpha, it - prev_iter, theta0=theta
        )
        prev_iter = it

        train_loss = cross_entropy_loss(X_train_gd, Y_train, theta)
        test_loss = cross_entropy_loss(X_test_gd, Y_test, theta)

        row = {
            "alpha": alpha,
            "iters": it,
            "Train Cross-Entropy": train_loss,
            "Test Cross-Entropy": test_loss
        }

        if it == 100:
            train_metrics = classification_metrics(X_train_gd, Y_train, theta)
            test_metrics = classification_metrics(X_test_gd, Y_test, theta)

            row.update({
                "Train Accuracy": train_metrics["Accuracy"],
                "Train Precision": train_metrics["Precision"],
                "Train Recall": train_metrics["Recall"],
                "Train F1": train_metrics["F1"],
                "Test Accuracy": test_metrics["Accuracy"],
                "Test Precision": test_metrics["Precision"],
                "Test Recall": test_metrics["Recall"],
                "Test F1": test_metrics["F1"]
            })

        rows.append(row)

results_df = pd.DataFrame(rows)
print(results_df)

   alpha  iters  Train Cross-Entropy  Test Cross-Entropy  Train Accuracy  \
0   0.01     10             0.651167            0.649058             NaN   
1   0.01     50             0.541620            0.534197             NaN   
2   0.01    100             0.468734            0.457877        0.897681   
3   0.10     10             0.465377            0.454300             NaN   
4   0.10     50             0.324124            0.306813             NaN   
5   0.10    100             0.289078            0.269988        0.907826   
6   0.50     10             0.320251            0.302551             NaN   
7   0.50     50             0.258894            0.238305             NaN   
8   0.50    100             0.243778            0.224939        0.918261   

   Train Precision  Train Recall  Train F1  Test Accuracy  Test Precision  \
0              NaN           NaN       NaN            NaN             NaN   
1              NaN           NaN       NaN            NaN             NaN   
2       

In [7]:
pkg_model = LogisticRegression(max_iter=5000)
pkg_model.fit(X_train_scaled, Y_train)

train_pred_pkg = pkg_model.predict(X_train_scaled)
test_pred_pkg = pkg_model.predict(X_test_scaled)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

package_results = pd.DataFrame([
    {
        "Model": "Package Logistic Regression",
        "Set": "Train",
        "Accuracy": accuracy_score(Y_train, train_pred_pkg),
        "Precision": precision_score(Y_train, train_pred_pkg, zero_division=0),
        "Recall": recall_score(Y_train, train_pred_pkg, zero_division=0),
        "F1": f1_score(Y_train, train_pred_pkg, zero_division=0)
    },
    {
        "Model": "Package Logistic Regression",
        "Set": "Test",
        "Accuracy": accuracy_score(Y_test, test_pred_pkg),
        "Precision": precision_score(Y_test, test_pred_pkg, zero_division=0),
        "Recall": recall_score(Y_test, test_pred_pkg, zero_division=0),
        "F1": f1_score(Y_test, test_pred_pkg, zero_division=0)
    }
])

print(package_results)

                         Model    Set  Accuracy  Precision    Recall        F1
0  Package Logistic Regression  Train  0.925797   0.923981  0.881166  0.902066
1  Package Logistic Regression   Test  0.925282   0.947126  0.867368  0.905495


The gradient descent model at alpha = 0.5 and 100 iterations is very similar to the package model, with just slightly lower F1 and recall scores.